# Assignment II – Bike Sharing Demand Prediction
**GitHub Repository:** Add Link

**Author:** Alesia Dako
**Date:** May 2026

## Task 1: Exploratory Data Analysis (EDA)

In this section, we explore the dataset to understand the structure of the data, the distribution of the target variable, and the relationships between features and bike rental counts. This analysis guides our feature engineering and modeling decisions.

In [2]:
%pip install numpy pandas matplotlib seaborn scikit-learn scikit-optimize xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 49.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 68.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 87.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 76.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 60.6 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 73.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 83.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 42.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 MB 41.0 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18/18 [scikit-optimize] [scikit-optimize]]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

from skopt import BayesSearchCV
from skopt.space import Real, Integer

import warnings
warnings.filterwarnings('ignore')

In [4]:
# Load dataset
df = pd.read_csv("data/hour.csv")

# Basic info
print(f"Dataset Shape: {df.shape}")
df.head()

EmptyDataError: No columns to parse from file

### 1.1 Dataset Overview

We begin by inspecting the shape, data types, and first rows of the dataset to understand what we are working with. The dataset contains 17,379 hourly records with no missing values, which simplifies preprocessing.

In [ ]:
# Target distribution
df['cnt'].describe()
print(f"Skewness: {df['cnt'].skew():.3f}")
plt.figure(figsize=(8,4))
sns.histplot(df['cnt'], kde=True)
plt.title("Distribution of cnt")
plt.show()

### 1.2 Dropping Irrelevant Columns

We drop `instant` (a row index with no predictive value), `dteday` (redundant since temporal information is already captured by `hr`, `mnth`, `yr`), and `casual` and `registered` (they are components of the target variable `cnt` and would cause data leakage if kept as features).

In [ ]:
# Drop irrelevant columns
df.drop(columns=['instant', 'dteday', 'casual', 'registered'], inplace=True)

### 1.3 Target Variable Distribution

The target variable `cnt` represents the total number of bike rentals per hour. Its distribution is right-skewed, meaning most hours have relatively low rental counts, with a long tail of high-demand hours. This skewness is expected in demand data and suggests that tree-based models may handle it better than linear ones, which assume normally distributed residuals.

In [ ]:
# Temporal patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), ['hr', 'weekday', 'mnth', 'season']):
    df.groupby(col)['cnt'].mean().plot(ax=ax, marker='o')
    ax.set_title(f'Avg cnt by {col}')
plt.tight_layout()
plt.show()

### Analytical Reasoning on Distribution:
- **Skewness:** The target variable cnt has a skewness of approximately 1.27 , confirming a strong right-skewed distribution. This suggests that while linear models might face challenges with non-normal residuals, tree-based models like Random Forest and Gradient Boosting are well-suited for this data.

- **Outliers:** The boxplot identifies several high-demand outliers. However, these are valid data points representing peak usage hours in Washington D.C. and will be retained to maintain the integrity of the business prediction problem. 

In [ ]:
# Skeweness
cnt_skew = df['cnt'].skew()
print(f"Target Variable Skewness: {cnt_skew:.3f}")

plt.figure(figsize=(10, 2))
sns.boxplot(x=df['cnt'])
plt.title("Boxplot of Bike Rental Count (cnt) for Outlier Identification")
plt.show()

### 1.4 Temporal Patterns

We analyze how average bike rentals vary across hours, weekdays, months, and seasons.

- **Hour (`hr`):** There are two clear peaks, around 8 AM and 5–6 PM, corresponding to morning and evening commute times. This is the strongest temporal signal in the data.
- **Weekday:** Weekdays show higher commute-driven peaks, while weekends show a flatter, more leisure-oriented distribution.
- **Month:** Demand is highest in summer months (June–September) and lowest in winter (December–February).
- **Season:** Summer and fall have the highest average rentals, confirming the monthly trend.

These patterns confirm that temporal features are highly predictive and must be carefully encoded.

### 1.5 Weather Feature Relationships

We examine how continuous weather variables relate to bike rental counts.

- **Temperature (`temp`) and Feeling Temperature (`atemp`):** Both show a positive relationship with `cnt` up to a point, after which very high temperatures reduce demand. `temp` and `atemp` are highly correlated, so we will consider dropping one to avoid multicollinearity.
- **Humidity (`hum`):** Higher humidity tends to reduce rentals, showing a mild negative relationship.
- **Wind Speed (`windspeed`):** Shows a weak negative relationship with rentals, though the effect is less pronounced than humidity.

In [ ]:
# Weatherfeaturesv sc n t
fi g ,axes=plt.subplots(1,4 ,figsize=(16,4 ) )
fora x ,c o li nzip(axes,I'temp','atemp', "hum','windspeed'1):
ax.scatter(df[coll, d f I ' c n t ' l , alpha=0.1, s=5) ax.set_xlabel(col)
ax.set_ylabel('cnt')
plt.tight_layout() p l t .show()

In [ ]:
# Exploring Potential Feature Interactions (Temp * Humidity)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='temp', y='cnt', hue='hum', palette='viridis', alpha=0.3, s=10)
plt.title("Interaction Exploration: Temperature vs. Count (colored by Humidity)")
plt.show()

### Justification for Interaction Terms:
- Analysis of the scatter plot suggests that the negative impact of high humidity on rentals becomes more pronounced as temperatures rise.

- This provides analytical justification for creating a temp * humidity interaction term in Task 3 to improve the model's predictive accuracy.